In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/daudaudinang/ui-elements-detection-dataset/dataset.yaml
/kaggle/input/datasets/daudaudinang/ui-elements-detection-dataset/README.md
/kaggle/input/datasets/daudaudinang/ui-elements-detection-dataset/.gitattributes
/kaggle/input/datasets/daudaudinang/ui-elements-detection-dataset/yolo_dataset/failed_urls.txt
/kaggle/input/datasets/daudaudinang/ui-elements-detection-dataset/yolo_dataset/scraper.log
/kaggle/input/datasets/daudaudinang/ui-elements-detection-dataset/yolo_dataset/statistics.json
/kaggle/input/datasets/daudaudinang/ui-elements-detection-dataset/yolo_dataset/classes.txt
/kaggle/input/datasets/daudaudinang/ui-elements-detection-dataset/yolo_dataset/annotated_images/mostvisited_tinyurl.com_1729630335_annotated.png
/kaggle/input/datasets/daudaudinang/ui-elements-detection-dataset/yolo_dataset/annotated_images/productivity_www.anydo.com_1729629633_annotated.png
/kaggle/input/datasets/daudaudinang/ui-elements-detection-dataset/yolo_dataset/annotated_images/off

In [2]:
# Cell 1 — Install
!pip install torchvision -q
import torch
import torchvision
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

✅ PyTorch: 2.9.0+cu126
✅ GPU: Tesla T4
✅ GPU Memory: 14.6 GB


In [3]:
# Cell 2 — Dataset class
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import torch

CLASS_NAMES = ["link","button","input","select","textarea","label",
               "checkbox","radio","dropdown","slider","toggle",
               "menu_item","clickable","icon","image","text"]

base = "/kaggle/input/datasets/daudaudinang/ui-elements-detection-dataset"

class UIDataset(Dataset):
    def __init__(self, img_dir, label_dir):
        self.img_dir   = img_dir
        self.label_dir = label_dir
        self.imgs = [f for f in os.listdir(img_dir)
                     if f.endswith(".png") or f.endswith(".jpg")]

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        lbl_path = os.path.join(self.label_dir,
                                img_name.rsplit(".", 1)[0] + ".txt")

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes  = []
        labels = []

        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls, xc, yc, bw, bh = map(float, parts)

                    # Convert YOLO → xyxy
                    x1 = (xc - bw/2) * w
                    y1 = (yc - bh/2) * h
                    x2 = (xc + bw/2) * w
                    y2 = (yc + bh/2) * h

                    if x2 <= x1 or y2 <= y1:
                        continue

                    boxes.append([x1, y1, x2, y2])
                    labels.append(int(cls) + 1)  # +1 for background class

        if len(boxes) == 0:
            boxes  = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,),   dtype=torch.int64)
        else:
            boxes  = torch.tensor(boxes,  dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}
        img    = TF.to_tensor(img)
        return img, target

def collate_fn(batch):
    return tuple(zip(*batch))

train_dataset = UIDataset(f"{base}/train/images", f"{base}/train/labels")
val_dataset   = UIDataset(f"{base}/val/images",   f"{base}/val/labels")
test_dataset  = UIDataset(f"{base}/test/images",  f"{base}/test/labels")

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False, collate_fn=collate_fn)

print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

✅ Train: 544 | Val: 71 | Test: 76


In [4]:
# Cell 3 — Build Faster RCNN model
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features  # fixed
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes + 1)
    return model

model = get_model(num_classes=16)
model = model.cuda()

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Faster RCNN ready")
print(f"✅ Parameters: {total_params:,}")

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 189MB/s] 


✅ Faster RCNN ready
✅ Parameters: 41,376,036


In [5]:
# Cell 4 — Training loop
from torch.optim.lr_scheduler import StepLR

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)
scheduler = StepLR(optimizer, step_size=10, gamma=0.5)

NUM_EPOCHS = 20
best_loss  = float("inf")

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    total_loss = 0

    for i, (images, targets) in enumerate(train_loader):
        images  = [img.cuda() for img in images]
        targets = [{k: v.cuda() for k, v in t.items()} for t in targets]

        # Skip batches with no annotations
        if all(len(t["boxes"]) == 0 for t in targets):
            continue

        loss_dict = model(images, targets)
        losses    = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += losses.item()

        if i % 30 == 0:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
                  f"Batch {i}/{len(train_loader)} | "
                  f"Loss: {losses.item():.4f}")

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    print(f"\n✅ Epoch {epoch+1} complete — Avg Loss: {avg_loss:.4f}\n")

    # Save best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(),
                   "/kaggle/working/faster_rcnn_best.pth")
        print(f"💾 Best model saved (loss: {best_loss:.4f})")

print("🎉 Training complete!")

Epoch 1/20 | Batch 0/136 | Loss: 11.5117
Epoch 1/20 | Batch 30/136 | Loss: 2.0293
Epoch 1/20 | Batch 60/136 | Loss: 1.4712
Epoch 1/20 | Batch 90/136 | Loss: 1.8497
Epoch 1/20 | Batch 120/136 | Loss: 1.7735

✅ Epoch 1 complete — Avg Loss: 2.2573

💾 Best model saved (loss: 2.2573)
Epoch 2/20 | Batch 0/136 | Loss: 1.4043
Epoch 2/20 | Batch 30/136 | Loss: 1.4642
Epoch 2/20 | Batch 60/136 | Loss: 1.9414
Epoch 2/20 | Batch 90/136 | Loss: 1.4752
Epoch 2/20 | Batch 120/136 | Loss: 2.0734

✅ Epoch 2 complete — Avg Loss: 1.5415

💾 Best model saved (loss: 1.5415)
Epoch 3/20 | Batch 0/136 | Loss: 1.9656
Epoch 3/20 | Batch 30/136 | Loss: 1.9949
Epoch 3/20 | Batch 60/136 | Loss: 1.0733
Epoch 3/20 | Batch 90/136 | Loss: 1.7126
Epoch 3/20 | Batch 120/136 | Loss: 1.6673

✅ Epoch 3 complete — Avg Loss: 1.4338

💾 Best model saved (loss: 1.4338)
Epoch 4/20 | Batch 0/136 | Loss: 1.2208
Epoch 4/20 | Batch 30/136 | Loss: 1.0267
Epoch 4/20 | Batch 60/136 | Loss: 1.7878
Epoch 4/20 | Batch 90/136 | Loss: 1.6453

In [6]:
# Cell 5 — Evaluate with mAP using torchmetrics
!pip install torchmetrics -q

from torchmetrics.detection.mean_ap import MeanAveragePrecision

model.eval()
metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True)

print("Evaluating on test set...")
with torch.no_grad():
    for images, targets in test_loader:
        images = [img.cuda() for img in images]

        outputs = model(images)

        # Format for torchmetrics
        preds = []
        for o in outputs:
            preds.append({
                "boxes":  o["boxes"].cpu(),
                "scores": o["scores"].cpu(),
                "labels": o["labels"].cpu() - 1,  # remove background offset
            })

        gts = []
        for t in targets:
            gts.append({
                "boxes":  t["boxes"],
                "labels": t["labels"] - 1,  # remove background offset
            })

        metric.update(preds, gts)

results = metric.compute()
print(f"\n✅ Evaluation complete")
print(f"mAP50:    {results['map_50']:.3f}")
print(f"mAP50-95: {results['map']:.3f}")

Evaluating on test set...

✅ Evaluation complete
mAP50:    0.215
mAP50-95: 0.129


In [7]:
# Cell 6 — Fixed per class results
print("Available keys:", list(results.keys()))

print(f"\n{'Class':<15} {'mAP50':>8}")
print("=" * 25)

# Get per class map correctly
if "map_per_class" in results:
    per_class = results["map_per_class"].tolist()
elif "map_50_per_class" in results:
    per_class = results["map_50_per_class"].tolist()
else:
    per_class = []

if len(per_class) > 0:
    for i, name in enumerate(CLASS_NAMES):
        if i < len(per_class):
            ap = per_class[i]
            if ap == -1:  # torchmetrics uses -1 for missing classes
                print(f"{name:<15} {'N/A':>8}")
            else:
                print(f"{name:<15} {ap:>8.3f}")
        else:
            print(f"{name:<15} {'N/A':>8}")
else:
    print("Per-class not available — using overall only")

print("=" * 25)
print(f"{'OVERALL':<15} {float(results['map_50']):>8.3f}")
print(f"{'mAP50-95':<15} {float(results['map']):>8.3f}")
print(f"{'Precision':<15} {float(results['map_75']):>8.3f}")
print(f"{'Recall':<15} {float(results['mar_100']):>8.3f}")

Available keys: ['map', 'map_50', 'map_75', 'map_small', 'map_medium', 'map_large', 'mar_1', 'mar_10', 'mar_100', 'mar_small', 'mar_medium', 'mar_large', 'map_per_class', 'mar_100_per_class', 'classes']

Class              mAP50
link               0.190
button             0.283
input              0.365
select             0.398
textarea           0.117
label              0.000
checkbox           0.000
radio              0.085
dropdown           0.000
slider             0.112
toggle             0.000
menu_item          0.000
clickable            N/A
icon                 N/A
image                N/A
text                 N/A
OVERALL            0.215
mAP50-95           0.129
Precision          0.142
Recall             0.170


In [8]:
# Cell 7 — Prediction function (same output format as YOLO)
import json

def predict(image_path, confidence=0.5):
    model.eval()
    img = Image.open(image_path).convert("RGB")
    img_tensor = TF.to_tensor(img).unsqueeze(0).cuda()

    with torch.no_grad():
        outputs = model(img_tensor)[0]

    detections = []
    for box, label, score in zip(
        outputs["boxes"], outputs["labels"], outputs["scores"]
    ):
        if score < confidence:
            continue
        x1, y1, x2, y2 = box.tolist()
        detections.append({
            "class":      CLASS_NAMES[label - 1],
            "bbox":       [round(x1), round(y1),
                           round(x2-x1), round(y2-y1)],
            "confidence": round(float(score), 2)
        })

    return detections

# Test on one image
test_imgs = os.listdir(f"{base}/test/images")
result = predict(f"{base}/test/images/{test_imgs[0]}")
print(json.dumps(result[:5], indent=2))

[
  {
    "class": "button",
    "bbox": [
      545,
      865,
      193,
      31
    ],
    "confidence": 0.99
  },
  {
    "class": "button",
    "bbox": [
      1417,
      413,
      165,
      37
    ],
    "confidence": 0.97
  },
  {
    "class": "button",
    "bbox": [
      38,
      855,
      375,
      37
    ],
    "confidence": 0.96
  },
  {
    "class": "link",
    "bbox": [
      328,
      117,
      165,
      23
    ],
    "confidence": 0.95
  },
  {
    "class": "button",
    "bbox": [
      1449,
      14,
      59,
      31
    ],
    "confidence": 0.92
  }
]


In [9]:
# Save model + full results
import torch, json

torch.save(
    model.state_dict(),
    "/kaggle/working/final_faster_rcnn_ui.pth"
)

faster_rcnn_results = {
    "model": "Faster RCNN ResNet50 FPN",
    "epochs": 20,
    "overall": {
        "mAP50":     float(results["map_50"]),
        "mAP50_95":  float(results["map"]),
        "precision": float(results["map_75"]),
        "recall":    float(results["mar_100"]),
    },
    "per_class": {
        CLASS_NAMES[i]: round(float(results["map_per_class"].tolist()[i]), 3)
        for i in range(len(results["map_per_class"].tolist()))
        if results["map_per_class"].tolist()[i] != -1
    }
}

with open("/kaggle/working/faster_rcnn_results.json", "w") as f:
    json.dump(faster_rcnn_results, f, indent=2)

print("✅ Model and results saved")
print(json.dumps(faster_rcnn_results, indent=2))

✅ Model and results saved
{
  "model": "Faster RCNN ResNet50 FPN",
  "epochs": 20,
  "overall": {
    "mAP50": 0.21509046852588654,
    "mAP50_95": 0.12917906045913696,
    "precision": 0.1417565941810608,
    "recall": 0.17017094790935516
  },
  "per_class": {
    "link": 0.19,
    "button": 0.283,
    "input": 0.365,
    "select": 0.398,
    "textarea": 0.117,
    "label": 0.0,
    "checkbox": 0.0,
    "radio": 0.085,
    "dropdown": 0.0,
    "slider": 0.112,
    "toggle": 0.0,
    "menu_item": 0.0
  }
}
